In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, cohen_kappa_score, classification_report, confusion_matrix, make_scorer
import seaborn as sns

for random forest think of it as you are making three models that use the previous models outputs as inputs for the next model. 

In this case the first random forest model would use d to predict A, the next one would use A, e, and f to predict B, and the final one would use A and B to predict C. 

Each sample is a sequence of 3 LSTM steps: A → B → C
following the Direct Parent Dependency rule.

Step 0 input: [d: employment, e: environmental]          → predict A: Supervisory

Step 1 input: [A_pred_embed, e: environmental, f: personnel] → predict B: Operator

Step 2 input: [B_pred_embed, A_pred_embed]                → predict C: Unsafe Act

Environmental = light + met + wind + temp (indices into num: wind=1, temp=2)

# Model 1

## 1. Load & Split Dataset

In [62]:
synthDataPath = "../../data/Simulated_Dataset.xlsx"
synthData = pd.read_excel(synthDataPath)
synthData

,Accident ID,Employment Change vs Prior Period (%),Light Conditions,Basic Meteorological Conditions,Wind Conditions (kt),Temperature (C),Personnel Conditions,Supervisory Conditions,Operator Conditions,Unsafe Conditions
0,ACC-0001,4.56,Daylight,VMC,21.1,27.7,Personal Readiness,Supervisory Violations,Physical/Mental Limitations,Skill-Based Errors
1,ACC-0002,11.41,Daylight,VMC,17.4,22.4,Crew Resource Management,Inadequate Supervision,Adverse Mental State,Decision Errors
2,ACC-0003,-8.32,Daylight,IMC,10.2,23.2,Personal Readiness,Supervisory Violations,Physical/Mental Limitations,Routine Violations
3,ACC-0004,-2.32,Daylight,VMC,37.5,-1.4,Personal Readiness,Planned Inappropriate Operations,Adverse Mental State,Exceptional Violations
4,ACC-0005,-3.85,Daylight,VMC,2.7,29.5,Personal Readiness,Planned Inappropriate Operations,Adverse Mental State,Exceptional Violations
...,...,...,...,...,...,...,...,...,...,...
495,ACC-0496,-10.91,Dusk,VMC,24.1,1.8,Crew Resource Management,Failed to Correct Problem,Adverse Physiological State,Skill-Based Errors
496,ACC-0497,-1.08,Night,VMC,2.1,7.9,Personal Readiness,Supervisory Violations,Adverse Physiological State,Decision Errors
497,ACC-0498,1.03,Daylight,IMC,4.4,5.0,Crew Resource Management,Planned Inappropriate Operations,Adverse Mental State,Perceptual Errors
498,ACC-0499,11.70,Daylight,IMC,4.1,7.7,Crew Resource Management,Failed to Correct Problem,Adverse Mental State,Decision Errors


In [ ]:
TEST_SPLIT = 0.2
RAND_STATE = 1024
n_test   = int(len(synthData) * TEST_SPLIT)
train_set, test_set = train_test_split(synthData,test_size=TEST_SPLIT, train_size=(1-TEST_SPLIT), random_state=RAND_STATE)

In [91]:
# TRAINING SET
envrCond_train = train_set[["Light Conditions","Basic Meteorological Conditions","Wind Conditions (kt)","Temperature (C)"]]
input_1_train = pd.concat((train_set["Employment Change vs Prior Period (%)"],envrCond_train),axis=1)
persCond_train = train_set["Personnel Conditions"]

y_A_train = train_set["Supervisory Conditions"]
y_B_train = train_set["Operator Conditions"]
y_C_train = train_set["Unsafe Conditions"]


# TESTING SET
envrCond_test = test_set[["Light Conditions","Basic Meteorological Conditions","Wind Conditions (kt)","Temperature (C)"]]
input_1_test = test_set[["Employment Change vs Prior Period (%)","Light Conditions","Basic Meteorological Conditions","Wind Conditions (kt)","Temperature (C)"]]
persCond_test = test_set["Personnel Conditions"]

y_A_test = test_set["Supervisory Conditions"]
y_B_test = test_set["Operator Conditions"]
y_C_test = test_set["Unsafe Conditions"]

input_1_train

,Employment Change vs Prior Period (%),Light Conditions,Basic Meteorological Conditions,Wind Conditions (kt),Temperature (C)
112,11.16,Daylight,VMC,3.4,20.8
44,-4.64,Night,IMC,1.1,14.0
187,6.71,Daylight,IMC,6.4,26.0
252,10.23,Daylight,IMC,3.6,7.4
102,-9.89,Daylight,VMC,16.6,14.2
...,...,...,...,...,...
101,-8.68,Daylight,VMC,0.7,5.0
492,3.40,Dusk,IMC,9.7,32.7
97,5.19,Daylight,VMC,8.2,12.2
401,10.84,Daylight,VMC,17.4,33.9


## 2. Create Pipeline & Classifier

In [12]:
forestParams = {
    "rf__n_estimators":[50,150,250],
    "rf__criterion":['gini','entropy','log_loss'],
    "rf__max_depth":[None,50,100],
    "rf__max_features":[None,'log2','sqrt'],
    "rf__min_samples_split":[2,4,8,16],
    "rf__min_samples_leaf":[2,4,8,16]
}

# Scoring metrics used - must use special variants adapted for multiple classes (recall, f1, etc. are normally binary)
scoring = {
    'accuracy':  'accuracy',
    'cohen_kappa': make_scorer(cohen_kappa_score),
    'precision': 'precision_macro',
    'recall':    'recall_macro',
    'f1':        'f1_macro',
    'roc_auc':   'roc_auc_ovr'
}

**Remember:**

Each sample is a sequence of 3 LSTM steps: A → B → C
following the Direct Parent Dependency rule.

Step 0 input: [d: employment, e: environmental]          → predict A: Supervisory

Step 1 input: [A_pred_embed, e: environmental, f: personnel] → predict B: Operator

Step 2 input: [B_pred_embed, A_pred_embed]                → predict C: Unsafe Act

Environmental = light + met + wind + temp (indices into num: wind=1, temp=2)

In [ ]:
class gridSearchForest(GridSearchCV):

    def __init__(self,input_set):
        self.input_set = input_set
        num_cols = self.input_set.select_dtypes(exclude=['object','str']).columns.tolist()
        cat_cols = self.input_set.select_dtypes(include=['object','str']).columns.tolist()

        encoder = ColumnTransformer([
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)``
        ],n_jobs=-1)

        forestClassifier = Pipeline([
            ('encoder', encoder),
            ('rf', RandomForestClassifier(random_state=RAND_STATE, bootstrap=True))
        ])

        super().__init__(
            estimator=forestClassifier,
            param_grid=forestParams,
            cv=5,
            scoring=scoring['recall'],
            refit=True,
            n_jobs=-1, #enables parallel processing. Here n_jobs=-1 means ALL available CPU cores
            verbose=1, #verbose=0 means silent mode (no output), verbose=1	means moderate output (Progress bar)
            return_train_score=True
            )

## 3. Train Model 1

In [ ]:
# Create and Fit Model 1
gridSearch_1 = gridSearchForest(input_1_train)
gridSearch_1.fit(input_1_train, y_A_train)

Fitting 5 folds for each of 1296 candidates, totalling 6480 fits


,input_set,Employme...s x 5 columns]
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_

In [ ]:
# Extract Best Estimator
newA_model = gridSearch_1.best_estimator_

# Out of Fold Predictions for prelim. eval & Model 1 Input
oof_1_pred = cross_val_predict(newA_model,input_1_train,y_A_train,cv=5,method='predict')


In [87]:
params1 = gridSearch_1.best_params_
print("Model 1 GridSearch Parameters:")
for key in params1.keys():
    print(f"\t{key} : {params1[key]}")

Model 1 GridSearch Parameters:
	rf__criterion : gini
	rf__max_depth : None
	rf__max_features : sqrt
	rf__min_samples_leaf : 2
	rf__min_samples_split : 2
	rf__n_estimators : 50


## 4. Evaluate Model 1

In [ ]:
# Generate test A for preliminary evaluation
newA_eval = newA_model.predict(input_1_test)

# Evaluate first model
newA_acc = accuracy_score(y_A_test, newA_eval)
newA_rec = recall_score(y_A_test, newA_eval, average='macro')
newA_f1 = f1_score(y_A_test, newA_eval, average='macro')
newA_coh = cohen_kappa_score(y_A_test, newA_eval)

print(f"""
      ---Model 1---
        Accuracy : {newA_acc}
        Recall   : {newA_rec}
        F1-Score : {newA_f1}
        Coh Kap  : {newA_coh}
      """)


      ---Model 1---
        Accuracy : 0.256
        Recall   : 0.2133288965185517
        F1-Score : 0.2130199118944044
        Coh Kap  : -0.048312554951867925
      


# Model 2

## 1. Data Setup

Note: Need to figure out how to put *predicted* A (currently 100 rows) into the second input set (400 rows)

In [ ]:
# Create Training and Test sets for Model 2 from data
input_2_train = pd.concat((envrCond_train,persCond_train),axis=1)
input_2_train['Supervisory Conditions'] = oof_1_pred
input_2_test = pd.concat((y_A_test,envrCond_test,persCond_test),axis=1)
input_2_train

,Light Conditions,Basic Meteorological Conditions,Wind Conditions (kt),Temperature (C),Personnel Conditions,Supervisory Conditions
112,Daylight,VMC,3.4,20.8,Crew Resource Management,Inadequate Supervision
44,Night,IMC,1.1,14.0,Personal Readiness,Inadequate Supervision
187,Daylight,IMC,6.4,26.0,Personal Readiness,Planned Inappropriate Operations
252,Daylight,IMC,3.6,7.4,Crew Resource Management,Planned Inappropriate Operations
102,Daylight,VMC,16.6,14.2,Crew Resource Management,Planned Inappropriate Operations
...,...,...,...,...,...,...
101,Daylight,VMC,0.7,5.0,Crew Resource Management,Failed to Correct Problem
492,Dusk,IMC,9.7,32.7,Crew Resource Management,Inadequate Supervision
97,Daylight,VMC,8.2,12.2,Personal Readiness,Inadequate Supervision
401,Daylight,VMC,17.4,33.9,Crew Resource Management,Inadequate Supervision


## 2. Fit and Train Model 2

In [48]:
# Greate and Fit Model 2
gridSearch_2 = gridSearchForest(input_2_train)
gridSearch_2.fit(input_2_train, y_B_train)

Fitting 5 folds for each of 1296 candidates, totalling 6480 fits


,input_set,Light Con...s x 6 columns]
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer

In [ ]:
# Extract Best Estimator
newB_model = gridSearch_2.best_estimator_
# gridSearch_2.best_params_

# Out of Fold Predictions for prelim. eval & Model 2 Input
oof_2_pred = cross_val_predict(newB_model, input_2_train, y_B_train,cv=5,method='predict')

# Generate new B for second model
# newB_pred = newB_model.predict(oof_2_pred)

In [ ]:
params2 = gridSearch_2.best_params_
print("Model 2 GridSearch Parameters:")
for key in params2.keys():
    print(f"\t{key} : {params2[key]}")

Model 3 GridSearch Parameters:
	rf__criterion : entropy
	rf__max_depth : None
	rf__max_features : None
	rf__min_samples_leaf : 2
	rf__min_samples_split : 16
	rf__n_estimators : 50


## 3. Evaluate Model 2

In [ ]:
# Generate test B for preliminary evaluation
newB_eval = newB_model.predict(input_2_test)


# Evaluate second model
newB_acc = accuracy_score(y_B_test, newB_eval)
newB_rec = recall_score(y_B_test, newB_eval, average='macro')
newB_f1 = f1_score(y_B_test, newB_eval, average='macro')
newB_coh = cohen_kappa_score(y_B_test, newB_eval)

print(f"""
      ---Model 2---
        Accuracy : {newB_acc}
        Recall   : {newB_rec}
        F1-Score : {newB_f1}
        Coh Kap  : {newB_coh}
      """)


      ---Model 1---
        Accuracy : 0.424
        Recall   : 0.3718081195092689
        F1-Score : 0.36086896911639177
        Coh Kap  : 0.07509698635768058
      


# Model 3

## 1. Data Setup

In [63]:
# Create Training and Test sets for Model 3 from data
input_3_train = pd.DataFrame({'Supervisory Conditions' : oof_1_pred,
                              'Operator Conditions' : oof_2_pred})
input_3_test = pd.concat((y_A_test,y_B_test),axis=1)

## 2. Fit and Train Model 3

In [64]:
# Greate and Fit Model 3
gridSearch_3 = gridSearchForest(input_3_train)
gridSearch_3.fit(input_3_train, y_C_train)

Fitting 5 folds for each of 1296 candidates, totalling 6480 fits


,input_set,...s x 2 columns]
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` a

In [65]:
# Extract Best Estimator
newC_model = gridSearch_3.best_estimator_
# gridSearch_2.best_params_

# Out of Fold Predictions for prelim. eval & Model 2 Input
oof_3_pred = cross_val_predict(newC_model, input_3_train, y_C_train,cv=5,method='predict')

# Generate new C for second model
# newC_pred = newC_model.predict(oof_3_pred)

In [78]:
params3 = gridSearch_3.best_params_
print("Model 3 GridSearch Parameters:")
for key in params3.keys():
    print(f"\t{key} : {params3[key]}")

Model 3 GridSearch Parameters:
	rf__criterion : gini
	rf__max_depth : None
	rf__max_features : log2
	rf__min_samples_leaf : 8
	rf__min_samples_split : 2
	rf__n_estimators : 50


## 3. Evaluate Model 3

In [ ]:
# Generate test C for preliminary evaluation
newC_eval = newC_model.predict(input_3_test)

# Evaluate first model
newC_acc = accuracy_score(y_C_test, newC_eval)
newC_rec = recall_score(y_C_test, newC_eval, average='macro')
newC_f1 = f1_score(y_C_test, newC_eval, average='macro')
newC_coh = cohen_kappa_score(y_C_test, newC_eval)

print(f"""
      ---Model 3---
        Accuracy : {newC_acc}
        Recall   : {newC_rec}
        F1-Score : {newC_f1}
        Coh Kap  : {newC_coh}
      """)


      ---Model 1---
        Accuracy : 0.26
        Recall   : 0.19583362583362582
        F1-Score : 0.1503786082733451
        Coh Kap  : 0.0006914135084913342
      


# Final Pipeline

In [73]:
A_pred = newA_model.predict(input_1_test)
input_2_test_final = input_2_test.copy()
input_2_test_final['Supervisory Conditions'] = A_pred
B_pred = newB_model.predict(input_2_test_final)
input_3_test_final = pd.DataFrame({'Supervisory Conditions' : A_pred,
                                   'Operator Conditions' : B_pred})
C_pred = newC_model.predict(input_3_test_final)

In [74]:
# Evaluate Final model
newC_acc = accuracy_score(y_C_test, C_pred)
newC_rec = recall_score(y_C_test, C_pred, average='macro')
newC_f1 = f1_score(y_C_test, C_pred, average='macro')
newC_coh = cohen_kappa_score(y_C_test, C_pred)

print(f"""
      ---Model 3---
        Accuracy : {newC_acc}
        Recall   : {newC_rec}
        F1-Score : {newC_f1}
        Coh Kap  : {newC_coh}
      """)


      ---Model 3---
        Accuracy : 0.248
        Recall   : 0.17897507897507897
        F1-Score : 0.12497342407648013
        Coh Kap  : -0.03392142197192993
      
